In [1]:
import pandas as pd

df = pd.read_csv(
    "../data/model_table.csv",
    dtype={"DIN": str, "AI_GROUP": str, "AI_GROUP_NO": str},
    parse_dates=["ACTUAL_START_DATE", "ACTUAL_END_DATE", "DATE_UPDATED", "REF_DATE"],
)

# The data snapshot date: the latest update in the export
snapshot = df["DATE_UPDATED"].max()

# Use every shortage that started at least 90 days before the snapshot,
# whether it is resolved or still ongoing. For all of them we already know
# whether the shortage lasted more than 90 days.
eligible = df[
    df["STATUS"].isin(["Resolved", "Actual shortage"])
    & df["ACTUAL_START_DATE"].notna()
    & (df["ACTUAL_START_DATE"] <= snapshot - pd.Timedelta(days=90))
].copy()

# Days known so far: until the end date, or until the snapshot if still ongoing
known_days = (eligible["ACTUAL_END_DATE"].fillna(snapshot) - eligible["ACTUAL_START_DATE"]).dt.days
eligible["TARGET"] = (known_days > 90).astype(int)

print(f"Snapshot date: {snapshot.date()}")
print(f"Shortages used: {len(eligible)}")
print(f"  of which still ongoing: {(eligible['STATUS'] == 'Actual shortage').sum()}")
print(f"Share lasting more than 90 days: {eligible['TARGET'].mean():.3f}")

Snapshot date: 2026-11-26
Shortages used: 22668
  of which still ongoing: 1247
Share lasting more than 90 days: 0.393


In [2]:
categorical_base = ["REASON", "MAIN_ROUTE", "ATC_GROUP", "COMPANY_NAME"]
numeric_base = ["ING_PRIOR_REPORTS", "COMPANY_PRIOR_REPORTS", "REPORTED_IN_ADVANCE"]

# Each feature set adds one group of features to the previous one
feature_sets = {
    "A: original features": (
        categorical_base,
        numeric_base,
    ),
    "B: + concurrent shortages": (
        categorical_base,
        numeric_base + ["CONCURRENT_SHORTAGES"],
    ),
    "C: + DPD market features": (
        categorical_base + ["MAIN_DOSAGE_FORM"],
        numeric_base + [
            "CONCURRENT_SHORTAGES",
            "N_SAME_INGREDIENT_ON_MARKET",
            "N_SAME_STRENGTH_ON_MARKET",
            "N_COMPANIES_SAME_INGREDIENT",
            "PRODUCT_AGE_YEARS",
            "IN_DPD",
        ],
    ),
}

# Time-based split: train on the earliest 80%, test on the most recent 20%
data = eligible.sort_values("ACTUAL_START_DATE").reset_index(drop=True)
cutoff = int(len(data) * 0.8)
train, test = data.iloc[:cutoff], data.iloc[cutoff:]
y_train, y_test = train["TARGET"], test["TARGET"]

print(f"Train: {len(train)} rows, long-shortage rate {y_train.mean():.3f}")
print(f"Test:  {len(test)} rows, long-shortage rate {y_test.mean():.3f}, "
      f"starting from {test['ACTUAL_START_DATE'].min().date()}")

Train: 18134 rows, long-shortage rate 0.396
Test:  4534 rows, long-shortage rate 0.381, starting from 2024-07-05


In [3]:
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


def make_models(categorical, numeric):
    """Build the three models for a given list of features."""
    encode = OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=20)
    encode_dense = OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=20,
                                 sparse_output=False)
    return {
        "Logistic regression": Pipeline([
            ("prep", ColumnTransformer([("cat", encode, categorical),
                                        ("num", StandardScaler(), numeric)])),
            ("clf", LogisticRegression(max_iter=2000, class_weight="balanced")),
        ]),
        "Gradient boosting": Pipeline([
            ("prep", ColumnTransformer([("cat", encode_dense, categorical),
                                        ("num", "passthrough", numeric)])),
            ("clf", HistGradientBoostingClassifier(class_weight="balanced", random_state=42)),
        ]),
        "Random forest": Pipeline([
            ("prep", ColumnTransformer([("cat", encode, categorical),
                                        ("num", "passthrough", numeric)])),
            ("clf", RandomForestClassifier(n_estimators=400, min_samples_leaf=5,
                                           class_weight="balanced", random_state=42, n_jobs=-1)),
        ]),
    }


def prepare(frame, categorical, numeric):
    """Select features and fill missing values."""
    X = frame[categorical + numeric].copy()
    X[categorical] = X[categorical].fillna("UNKNOWN")
    X[numeric] = X[numeric].fillna(-1)  # -1 marks "not available", e.g. product not in DPD
    return X


results = []
fitted = {}
for set_name, (categorical, numeric) in feature_sets.items():
    X_train = prepare(train, categorical, numeric)
    X_test = prepare(test, categorical, numeric)
    for model_name, model in make_models(categorical, numeric).items():
        model.fit(X_train, y_train)
        auc = roc_auc_score(y_test, model.predict_proba(X_test)[:, 1])
        results.append({"features": set_name, "model": model_name, "roc_auc": round(auc, 3)})
        fitted[(set_name, model_name)] = model
        print(f"done: {set_name} / {model_name}  AUC = {auc:.3f}")

ablation = pd.DataFrame(results).pivot(index="features", columns="model", values="roc_auc")
ablation

done: A: original features / Logistic regression  AUC = 0.667
done: A: original features / Gradient boosting  AUC = 0.668
done: A: original features / Random forest  AUC = 0.694
done: B: + concurrent shortages / Logistic regression  AUC = 0.678
done: B: + concurrent shortages / Gradient boosting  AUC = 0.672
done: B: + concurrent shortages / Random forest  AUC = 0.698
done: C: + DPD market features / Logistic regression  AUC = 0.677
done: C: + DPD market features / Gradient boosting  AUC = 0.695
done: C: + DPD market features / Random forest  AUC = 0.707


model,Gradient boosting,Logistic regression,Random forest
features,,,
A: original features,0.668,0.667,0.694
B: + concurrent shortages,0.672,0.678,0.698
C: + DPD market features,0.695,0.677,0.707


In [4]:
from sklearn.metrics import precision_score, recall_score

best_categorical, best_numeric = feature_sets["C: + DPD market features"]
best_model = fitted[("C: + DPD market features", "Random forest")]
X_test_best = prepare(test, best_categorical, best_numeric)
proba = best_model.predict_proba(X_test_best)[:, 1]

# Try different probability thresholds for flagging a shortage as "long"
rows = []
for threshold in [0.3, 0.35, 0.4, 0.45, 0.5, 0.55, 0.6]:
    pred = (proba >= threshold).astype(int)
    rows.append({
        "threshold": threshold,
        "flagged_share": round(pred.mean(), 3),
        "recall_long": round(recall_score(y_test, pred), 3),
        "precision_long": round(precision_score(y_test, pred), 3),
    })
pd.DataFrame(rows)

,threshold,flagged_share,recall_long,precision_long
0,0.30,0.856,0.946,0.422
1,0.35,0.714,0.871,0.465
2,0.40,0.581,0.760,0.498
3,0.45,0.461,0.655,0.541
4,0.50,0.342,0.515,0.575
5,0.55,0.226,0.362,0.610
6,0.60,0.134,0.234,0.664


In [5]:
from sklearn.inspection import permutation_importance

# Shuffle one feature at a time and measure how much the test AUC drops
perm = permutation_importance(
    best_model, X_test_best, y_test,
    scoring="roc_auc", n_repeats=5, random_state=42, n_jobs=-1,
)

importance = (
    pd.DataFrame({
        "feature": X_test_best.columns,
        "auc_drop": perm.importances_mean.round(4),
        "std": perm.importances_std.round(4),
    })
    .sort_values("auc_drop", ascending=False)
    .reset_index(drop=True)
)
importance

,feature,auc_drop,std
0,COMPANY_NAME,0.0603,0.0045
1,COMPANY_PRIOR_REPORTS,0.0424,0.0043
2,REASON,0.0381,0.0036
3,CONCURRENT_SHORTAGES,0.0162,0.0016
4,PRODUCT_AGE_YEARS,0.0134,0.0013
5,ATC_GROUP,0.0061,0.0013
6,N_SAME_STRENGTH_ON_MARKET,0.0049,0.0013
7,ING_PRIOR_REPORTS,0.0037,0.0006
8,MAIN_DOSAGE_FORM,0.0033,0.0008
9,N_COMPANIES_SAME_INGREDIENT,0.0017,0.0013


## Results with full data (2017 to 2026) and DPD features

**Target fix:** shortages that are still ongoing but already longer than 90 days are now labelled as long. This removed the censoring bias: the long-shortage rate is 39.6% in train and 38.1% in test (before the fix: 34% vs 21%).

**Ablation (ROC AUC on shortages starting from July 2024):**

| Features | Logistic regression | Gradient boosting | Random forest |
|---|---|---|---|
| A: original features | 0.667 | 0.668 | 0.694 |
| B: + concurrent shortages | 0.678 | 0.672 | 0.698 |
| C: + DPD market features | 0.677 | 0.695 | 0.707 |

- Each feature group adds a small, real gain (about 0.01 to 0.03 AUC).
- AUC stays around 0.70. Much of what decides a shortage's length (repair time, raw material supply) is not known when the shortage is reported.

**Chosen model:** random forest with feature set C, threshold 0.40. It catches 76% of long shortages, and 50% of flagged shortages are truly long (38% if flagged at random). The threshold was chosen on the test set; a separate validation set would be more rigorous.

**What drives the model (permutation importance):** company name and the company's past shortage reports matter most, followed by reason, concurrent shortages of the same ingredient, and product age. Predictions for new or rarely seen companies will be less reliable.